# Traffic flow nowcasting with Ultralytics YOLO26 (GPU version)

Runs the whole project on your own GPU (tested settings for an **RTX 4090**):
1. Download the 3 public traffic videos and the KITTI dataset
2. YOLO26 detection + ByteTrack tracking on the videos
3. Counting, inverse-depth time-to-line forecasts, adaptive conformal intervals, next-3 s flow nowcast, plus figures
4. Fine-tune YOLO26 on the **full** KITTI dataset
5. Render the ~5-minute results video

Open this notebook from inside the project folder (the one with `track.py`). Each cell can be re-run on its own.

In [ ]:
# 1) install and check the GPU
%pip install -q -r requirements.txt
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
# If CUDA is False, install a CUDA build of PyTorch first: https://pytorch.org/get-started/locally/

In [ ]:
# 2) download data (works on Windows, Linux and macOS)
import os, urllib.request, zipfile
os.makedirs("data", exist_ok=True); os.makedirs("datasets", exist_ok=True)
files = {
    "data/bay_traffic.mp4": "https://raw.githubusercontent.com/Bayunova28/Vehicle_Counting_Detection/master/Traffic.mp4",
    "data/bridge.mp4": "https://raw.githubusercontent.com/MicrocontrollersAndMore/OpenCV_3_Car_Counting_Cpp/master/CarsDrivingUnderBridge.mp4",
    "data/az_input.mp4": "https://raw.githubusercontent.com/ahmetozlu/vehicle_counting_tensorflow/master/input_video.mp4",
    "datasets/kitti.zip": "https://github.com/ultralytics/assets/releases/download/v0.0.0/kitti.zip",
}
for path, url in files.items():
    if not os.path.exists(path):
        print("downloading", path); urllib.request.urlretrieve(url, path)
if not os.path.exists("datasets/kitti/images"):
    zipfile.ZipFile("datasets/kitti.zip").extractall("datasets/kitti")
print(os.listdir("datasets/kitti"))

In [ ]:
# 3) detection + tracking on the GPU (a bigger detector than the CPU demo)
import os, subprocess, sys
os.environ["TRACK_MODEL"] = "yolo26m.pt"     # yolo26s.pt is faster, yolo26l.pt / yolo26x.pt are more accurate
for video, out, secs in [("data/bay_traffic.mp4", "tracks_bay.json", 81),
                         ("data/bridge.mp4", "tracks_bridge.json", 33),
                         ("data/az_input.mp4", "tracks.json", 30)]:
    subprocess.run([sys.executable, "track.py", video, out, str(secs)], check=True)

In [ ]:
# 4) counting + forecasting analysis + figures  (writes results/*.json and figs/*.png)
import glob, os, subprocess, sys, json
for f in glob.glob("results/cache_*.pkl"): os.remove(f)      # clear the cache from earlier runs
subprocess.run([sys.executable, "figs_analysis.py"], check=True, stdout=subprocess.DEVNULL)
m = json.load(open("results/forecast_metrics.json"))
for k, v in m.items():
    print(f"{k:12s} crossings={v['crossings']:3d}  TTL MAE ours={v['ttl_mae_all_s']['ours_inverse_depth']}s "
          f"(const-accel {v['ttl_mae_all_s']['const_accel']}s)  coverage={v['conformal']['coverage']:.0%}  "
          f"3s-count MAE ours={v['nowcast']['mae_ours']} vs persistence {v['nowcast']['mae_persistence']}")

In [ ]:
# 5) fine-tune YOLO26 on the FULL KITTI dataset (RTX 4090: roughly 1-2 h for 100 epochs at 640 px)
import os, subprocess, sys, json, shutil
root = os.path.abspath("datasets/kitti")
names = ["car", "van", "truck", "pedestrian", "person_sitting", "cyclist", "tram", "misc"]
open("kitti_full.yaml", "w").write(f"path: {root}\ntrain: images/train\nval: images/val\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(names)))
EPOCHS, IMGSZ, MODEL, BATCH = 100, 640, "yolo26s.pt", 32    # try EPOCHS=30 for a quick run, or MODEL="yolo26m.pt"
os.makedirs("train", exist_ok=True)
subprocess.run([sys.executable, "../train.py", "../kitti_full.yaml", str(EPOCHS), str(IMGSZ), MODEL, "0", str(BATCH)], check=True, cwd="train")
ts = json.load(open("train/train_summary.json"))
ts["data_desc"] = "Full KITTI · 5,985 train / 1,496 val · 8 classes"; ts["val_images"] = 1496
json.dump(ts, open("train/train_summary.json", "w"), indent=2); shutil.copy("train/train_summary.json", "results/train_summary.json")
print(json.dumps(ts["metrics"], indent=1))

In [ ]:
# 6) render the results video (about 5 minutes, 1920x1080) and the README
import os, subprocess, sys
if os.path.exists("figs/fig_train_curves.png"): os.remove("figs/fig_train_curves.png")   # rebuild from the new training run
subprocess.run([sys.executable, "build_video.py", "render", *map(str, range(22))], check=True)
subprocess.run([sys.executable, "build_video.py", "concat", "traffic_nowcasting_yolo26.mp4"], check=True)
subprocess.run([sys.executable, "make_readme.py"], check=True)
print("video: traffic_nowcasting_yolo26.mp4 | README: README_repo.md")

In [ ]:
# 7) preview
from IPython.display import Video, Image, display
display(Image("figs/fig_ttl_mae.png", width=900))
display(Video("traffic_nowcasting_yolo26.mp4", width=900, embed=False))

**Notes**
- The video needs `ffmpeg` on your PATH (`conda install -c conda-forge ffmpeg`).
- To use your own road footage, add it to `configs.py` with a counting line `(y, x_min, x_max)` in source pixels, then rerun cells 3, 4 and 6.
- If you edit `configs.py`, keep the 3 video keys or update `build_video.py` to match.